# CorefInst — Step-by-Step Documentation

This notebook explains every component of the CorefInst pipeline, how each piece works, and how to run it end-to-end.

---

## Table of Contents

1. [Overview](#1-overview)
2. [Repository Layout](#2-repository-layout)
3. [Installation](#3-installation)
4. [Dataset and CoNLL Parsing](#4-dataset-and-conll-parsing)
5. [Preprocessing — FrameExamples](#5-preprocessing--frameexamples)
6. [Training](#6-training)
7. [Inference](#7-inference)
8. [Evaluation Metrics](#8-evaluation-metrics)
9. [Baselines](#9-baselines)
10. [Running on Google Colab](#10-running-on-google-colab)
11. [Configuration Reference](#11-configuration-reference)
12. [Troubleshooting](#12-troubleshooting)

---
## 1. Overview

**CorefInst** (TACL 2026) approaches coreference resolution as a *structured generation* task.

### Key idea
Given a document where all mention spans are already identified (gold mentions), the model must assign a **cluster number** to each mention. Mentions with the same cluster number are coreferent.

### Why instruction-tuning?
Instead of a classification head or span-pair scorer, CorefInst uses a decoder-only LLM prompted with a natural-language instruction. The model generates one integer per mention — the cluster ID — using **controlled inference**: it predicts each MASK token sequentially, feeding each answer back as context before predicting the next.

### Languages
This implementation covers **Hindi (hi)**, **Tamil (ta)**, and **Bengali (bn)** from the TransMuCoRes dataset.

### Pipeline summary
```
TransMuCoRes CoNLL files
        │
        ▼
  conll_parser.py          → Document objects (sentences, mentions, clusters)
        │
        ▼
  preprocessor.py          → FrameExample objects (masked text + expected cluster numbers)
        │
        ▼
  dataset_builder.py       → HuggingFace Dataset (formatted with chat template)
        │
        ▼
  train.py (SFTTrainer)    → Fine-tuned LoRA adapter saved to model_output/
        │
        ▼
  inference.py             → Controlled generation → local cluster predictions
        │
        ▼
  postprocessor.py         → Algorithm 1: merge local clusters → global document clusters
        │
        ▼
  evaluate.py              → MUC / B³ / CEAFe / CoNLL-F scores
```

---
## 2. Repository Layout

```
.
├── src/
│   ├── conll_parser.py       # Parse CoNLL → Document
│   ├── preprocessor.py       # Document → FrameExample
│   ├── dataset_builder.py    # Load all languages + build HF Dataset
│   ├── model.py              # Load LLM + LoRA (unsloth or PEFT)
│   ├── train.py              # SFTTrainer loop
│   ├── inference.py          # Controlled generation
│   ├── postprocessor.py      # Algorithm 1 — cross-frame cluster merging
│   └── evaluate.py           # MUC, B³, CEAFe
├── analysis/
│   ├── baseline.py           # Simple baselines (no model)
│   └── analyse_results.py    # Deep-dive analysis of results.json
├── configs/
│   ├── t4.yaml               # T4 GPU preset (Colab free)
│   ├── a100.yaml             # A100 GPU preset (Colab Pro)
│   └── cpu.yaml              # CPU-only (Qwen 1.5B)
├── prepare_data.py           # Entry point: CoNLL → JSONL
├── train_model.py            # Entry point: JSONL → fine-tuned adapter
├── run_inference.py          # Entry point: adapter → predictions → scores
├── CorefInst_Colab.ipynb     # All-in-one Colab notebook
├── package_for_colab.py      # Zips the code for Drive upload
├── config.yaml               # Default config
└── requirements.txt
```

---
## 3. Installation

### Local machine (GPU recommended)

```bash
pip install -r requirements.txt
```

For faster training via **unsloth** (recommended on Colab):
```bash
pip install "unsloth[colab-new]"
```

For **Flash Attention 2** (optional, Ampere+ GPUs only):
```bash
pip install flash-attn --no-build-isolation
```

### Key packages

| Package | Purpose |
|---|---|
| `transformers` | Model loading, tokenization |
| `peft` | LoRA adapters |
| `trl` | SFTTrainer for instruction fine-tuning |
| `bitsandbytes` | 4-bit quantization |
| `datasets` | HuggingFace Dataset API |
| `scipy` | Hungarian algorithm for CEAFe |
| `unsloth` | (optional) Faster QLoRA kernels |

---
## 4. Dataset and CoNLL Parsing

### TransMuCoRes sub-datasets

The dataset lives under `transmucores_data/` and has three parts:

| Sub-dataset | Languages used | Format | Splits |
|---|---|---|---|
| `mujadia_conll/` | Hindi only | `*_gold_conll` files | train / development / test |
| `onto_notes_archive/` | Hi + Ta + Bn (with other languages) | `.conll` files, recursive | train / development / test |
| `litbank_{train,val,test}/` | Hi + Ta + Bn (with other languages) | `.conll` files, flat | train / val / test |

Language codes used as filename filters:
- Hindi: `hin_Deva`
- Tamil: `tam_Taml`
- Bengali: `ben_Beng`

### CoNLL format

Files are tab-separated with ~17 columns. The **last column** contains coreference annotations:

| Annotation | Meaning |
|---|---|
| `-` | Token not in any mention |
| `(N)` | Singleton mention of cluster N |
| `(N` | Start of multi-token mention of cluster N |
| `N)` | End of multi-token mention of cluster N |
| `(N)\|(M)` | Two overlapping mentions at the same token |

### Parsing example

In [ ]:
import sys; sys.path.insert(0, ".")
from src.conll_parser import load_conll_dir

docs = load_conll_dir(
    "transmucores_data/litbank_test",
    language_filter=["hin_Deva"],  # only Hindi files
    language="hi",
)

doc = docs[0]
print(f"Doc ID : {doc.doc_id}")
print(f"Sentences : {len(doc.sentences)}")
print(f"Mentions  : {len(doc.mentions)}")
print(f"Clusters  : {len(doc.clusters)}")

# Inspect first sentence
sent = doc.sentences[0]
print(f"\nSentence 0: {sent.plain_text()}")

# Inspect first mention
m = doc.mentions[0]
print(f"\nFirst mention: sent={m.sent_idx}, tokens [{m.start_tok}:{m.end_tok}], cluster={m.cluster_id}")

---
## 5. Preprocessing — FrameExamples

### Frame packing

The preprocessor groups consecutive sentences into **frames** of at most `max_tokens_per_frame` tokens (default 256). Each training example uses **two consecutive frames** joined by the separator `[MID]`.

Mentions within a frame are wrapped with `<m>…</m>` and followed by `#MASK` — a placeholder that the model must replace with a cluster number during inference.

**Input format (simplified):**
```
[Instruction]

She<m>said</m>#MASK she would come. [MID] <m>She</m>#MASK arrived on time.
```

**Expected output:**
```
1 1
```
(Both mentions belong to cluster 1.)

### Instruction #5 (default, best-performing)

The instruction explains that mentions sharing a cluster are coreferential AND coherent (not just similar). It includes a non-example ("author and book represent different entities") to help the model avoid over-merging.

### Preprocessing pipeline

In [ ]:
from src.preprocessor import create_frame_examples

# doc was loaded above
examples = create_frame_examples(doc, instruction_id=5, max_tokens_per_frame=256)
print(f"{len(examples)} FrameExamples from document '{doc.doc_id}'")

ex = examples[0]
print("\n--- Input (first 400 chars) ---")
print(ex.masked_input[:400])
print("\n--- Expected output ---")
print(ex.expected_output)

### Step 1: Prepare all data

Run this once. It reads all CoNLL files and writes JSONL files to `processed_data/`.

In [ ]:
# From terminal:
# python prepare_data.py --config config.yaml

# Or programmatically:
import subprocess
result = subprocess.run(["python", "prepare_data.py", "--config", "config.yaml"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

---
## 6. Training

### What happens during training

- The model is loaded with **4-bit quantization** (QLoRA) to fit in GPU memory.
- A **LoRA adapter** (rank=16) is attached to the attention and MLP projection layers.
- The `SFTTrainer` from TRL computes loss **only on the assistant response tokens** (the cluster number outputs), not on the instruction or input.
- Training runs for 3 epochs with a cosine learning rate schedule.

### Choosing the right config

| Your GPU | Use config |
|---|---|
| A100 (40/80 GB) | `config.yaml` or `configs/a100.yaml` |
| T4 (16 GB, Colab free) | `configs/t4.yaml` |
| No GPU / testing | `configs/cpu.yaml` (Qwen 1.5B, slow) |

### Running training

In [ ]:
# Standard training
# python train_model.py --config configs/t4.yaml

# Few-shot (N examples per language — good for quick iteration)
# python train_model.py --config configs/t4.yaml --few_shot 50

# The adapter is saved to model_output/final/
# Checkpoints every 200 steps in model_output/checkpoint-*/
print("See train_model.py for the entry point.")
print("Output adapter saved to: model_output/final/")

### LoRA configuration

Default LoRA settings (from the paper):

| Parameter | Value |
|---|---|
| Rank (r) | 16 |
| Alpha | 16 |
| Dropout | 0.0 |
| Target modules | q, k, v, o projections + gate/up/down MLP |

These can be changed in `config.yaml` under the `lora:` section.

---
## 7. Inference

### Controlled inference (Section 3.3 of the paper)

Standard generation would let the model output all cluster numbers at once — but that risks hallucinating inconsistent assignments. **Controlled inference** predicts cluster numbers one at a time:

1. Split the masked input on each `#MASK` token.
2. For each MASK position:
   - Build the prompt up to that point (including all previous predictions).
   - Generate `max_new_tokens=4` (enough for a 3-digit cluster number).
   - Extract the leading integer from the generation.
   - Append that number to the running context.
3. Collect all predicted cluster numbers for the current frame pair.

### Cross-frame cluster merging (Algorithm 1)

Each document is split into overlapping frame pairs. The same mention may appear in two consecutive pairs. Algorithm 1 in `postprocessor.py` uses these overlapping mention positions to map per-frame local cluster numbers to a consistent global document-level cluster assignment.

### Running inference

In [ ]:
# python run_inference.py --config configs/t4.yaml
# 
# Outputs:
#   - Per-language CoNLL scores printed to stdout
#   - inference_output/results.json  (machine-readable scores)
print("See run_inference.py for the entry point.")
print("Results saved to: inference_output/results.json")

### Viewing results

In [ ]:
import json, os

results_path = "inference_output/results.json"
if os.path.exists(results_path):
    with open(results_path) as fh:
        results = json.load(fh)
    for lang, r in results.items():
        print(f"{lang.upper():10s} MUC={r['muc']['f']:.2f}  B3={r['b3']['f']:.2f}  "
              f"CEAFe={r['ceafe']['f']:.2f}  CoNLL={r['conll']['f']:.2f}")
else:
    print("Run inference first: python run_inference.py --config config.yaml")

---
## 8. Evaluation Metrics

All metrics are computed as micro-averaged F1 across all documents.

### MUC (Vilain et al., 1995)
Counts the minimum number of links needed to partition the predicted clusters to match the gold clusters. Favours recall.

### B³ (Bagga & Baldwin, 1998)
For each mention, measures the overlap between its predicted cluster and its gold cluster. Penalises both over-merging and under-merging.

### CEAFe (Luo, 2005)
Finds the optimal one-to-one alignment between gold and predicted clusters using the Hungarian algorithm (or greedy if scipy is unavailable). Uses the phi_4 similarity: `2|G∩P| / (|G| + |P|)`.

### CoNLL-F
The official CoNLL shared task score: `(MUC-F + B³-F + CEAFe-F) / 3`.

### Running metrics directly

In [ ]:
from src.evaluate import evaluate_documents, print_scores
from src.postprocessor import extract_gold_clusters

# Example: oracle predictions (gold = prediction → should give ~100%)
if 'docs' in dir() and docs:
    gold_list, pred_list = [], []
    for doc in docs[:3]:
        _, gc = extract_gold_clusters(doc)
        gold_list.append(gc)
        pred_list.append(gc)  # oracle: predict gold exactly

    scores = evaluate_documents(gold_list, pred_list)
    print_scores(scores, label="Oracle (gold=pred)")
else:
    print("Load docs first (see Section 4).")

---
## 9. Baselines

Three rule-based baselines are included for comparison.

| Baseline | Description | Typical CoNLL-F |
|---|---|---|
| All-singletons | Every mention = own cluster | ~24 |
| All-one-cluster | All mentions in one cluster | ~40 |
| MFE | Same head word → same cluster | ~61 |

Run all baselines on the test split:

In [ ]:
# python analysis/baseline.py --config config.yaml --split test

import subprocess
result = subprocess.run(
    ["python", "analysis/baseline.py", "--config", "config.yaml", "--split", "test"],
    capture_output=True, text=True
)
print(result.stdout[:3000])
if result.returncode != 0:
    print(result.stderr[:1000])

---
## 10. Running on Google Colab

### Recommended workflow

**Step 1** — Package the code locally:
```bash
python package_for_colab.py
# Creates: corefinst_code.zip
```

**Step 2** — Upload to Google Drive:
- `corefinst_code.zip`
- `transmucores_data.tar.gz`

Both files go in the same Drive folder (default: `MyDrive/CorefInst`).

**Step 3** — Open `CorefInst_Colab.ipynb` in Colab and **Run All**.

The notebook will:
1. Detect GPU and apply the correct hardware preset (T4 or A100).
2. Mount Google Drive.
3. Install dependencies (`unsloth`, `trl`, etc.).
4. Extract code and dataset.
5. Update config paths to point at Drive.
6. Run `prepare_data.py` → `train_model.py` → `run_inference.py`.
7. Print results and save adapter + results to Drive.

### Colab tips

- **Session disconnects**: outputs and `model_output/` are saved to Drive after each step — resuming is safe.
- **T4 vs A100**: T4 (free tier) takes ~3h for 3 epochs. A100 (Pro) takes ~40min.
- **Few-shot mode**: set `FEW_SHOT = 50` in the notebook to train on 50 examples per language for quick experimentation.
- **Out of memory**: reduce `batch_size` in the config or switch to `configs/t4.yaml`.

---
## 11. Configuration Reference

All settings live in `config.yaml` (or a preset in `configs/`). Key fields:

```yaml
data:
  root: "transmucores_data"    # path to raw dataset
  output_dir: "processed_data" # where to write train/dev/test.jsonl
  languages:
    hi: {name: "Hindi",   codes: ["hin_Deva"]}
    ta: {name: "Tamil",   codes: ["tam_Taml"]}
    bn: {name: "Bengali", codes: ["ben_Beng"]}

preprocessing:
  instruction_id: 5           # 1-5; 5 is best per Table 1 of the paper
  max_tokens_per_frame: 256   # approx tokens per frame window

model:
  name: "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # or any HF model ID
  max_seq_length: 4096        # 2048 for T4, 4096 for A100
  load_in_4bit: true

lora:
  rank: 16
  alpha: 16
  dropout: 0.0

training:
  num_epochs: 3
  batch_size: 4               # reduce to 2 on T4
  gradient_accumulation_steps: 4
  learning_rate: 2.0e-4
  bf16: true                  # use fp16: true on T4

inference:
  max_cluster_id: 200
  output_dir: "inference_output"
```

---
## 12. Troubleshooting

### `ModuleNotFoundError: No module named 'datasets'`
```bash
pip install datasets
```

### `0 documents loaded` for a language
Check that the dataset path in `config.yaml → data.root` points to the extracted `transmucores_data/` directory, and that the directory contains the expected sub-folders (`litbank_test/`, `onto_notes_archive/`, etc.).

### CUDA out of memory during training
- Switch to `configs/t4.yaml` (batch_size=2, seq_len=2048).
- Or reduce `max_tokens_per_frame` in config (fewer tokens per training example).
- Or use `--few_shot 20` to train on fewer examples.

### MUC score > 100% (metric sanity check)
This indicates a mention position key mismatch between the preprocessor and evaluator. The fix is already applied in `preprocessor.py` — make sure you have the latest version.

### Training loss doesn't decrease
- Verify the response template is being detected correctly. Check `src/train.py → _find_response_template`.
- Ensure `load_in_4bit=true` and LoRA is applied (check logs for "trainable parameters").

### Inference produces no predictions
Check that the `#MASK` tokens are present in the processed examples. Run a quick sanity check:
```python
from src.preprocessor import create_frame_examples
from src.conll_parser import load_conll_dir
docs = load_conll_dir("transmucores_data/litbank_test", language_filter=["hin_Deva"], language="hi")
exs = create_frame_examples(docs[0])
print(exs[0].masked_input.count("#MASK"), "masks in first example")
```

### Slow inference
Controlled inference is sequential by design — each MASK prediction is one forward pass. For large test sets this is slow. Consider reducing the number of test documents or running with `--split dev` for quick checks.

---
## Full pipeline (one-shot execution)

If you just want to run everything from scratch in order:

In [ ]:
import subprocess, sys

CONFIG = "configs/t4.yaml"  # change to config.yaml for A100

steps = [
    ["python", "prepare_data.py",  "--config", CONFIG],
    ["python", "train_model.py",   "--config", CONFIG],
    ["python", "run_inference.py", "--config", CONFIG],
    ["python", "analysis/baseline.py", "--config", CONFIG, "--split", "test"],
    ["python", "analysis/analyse_results.py",
     "--results_json", "inference_output/results.json"],
]

for cmd in steps:
    print("\n" + "="*60)
    print("Running:", " ".join(cmd))
    print("="*60)
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print(f"ERROR: step failed with code {result.returncode}")
        sys.exit(result.returncode)

print("\nDone!")